# Fetching Facility Data

Reverse engineering Yale's energy usage website

In [1]:
import json
import re

import polars as pl
import requests
from bs4 import BeautifulSoup

In [2]:
response = requests.get("https://java.facilities.yale.edu/energy/")

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

script_text = soup.find("script", string=re.compile("var user")).string

# extract JSON parts
user_json = re.search(r'var user\s*=\s*(\{.*?\});', script_text, re.S).group(1)
buildings_json = re.search(r'var buildings\s*=\s*(\[.*?\]);', script_text, re.S).group(1)

user = json.loads(user_json)
buildings = json.loads(buildings_json)

# print(user)
# print(buildings)

In [4]:
rows = []

for s in user["sets"]:
    set_meta = {
        "bsID": s["bsID"],
        "bsName": s["bsName"],
        "bsDescription": s.get("bsDescription"),
        "bsPublic": s["bsPublic"],
        "owner": s["owner"],
        "sort": s["sort"]
    }

    for b in s["buildings"]:
        row = {**set_meta, **b}
        rows.append(row)

df = pl.DataFrame(rows)

In [5]:
facilities = (
    df
    .unique(subset=["facid"])
    .select([
        "facid",
        "site",
        "name",
        "address",
        "zone",
        "owned",
        "status",
        "latitude",
        "longitude"
    ])
    .sort("facid")
)
building_sets = (
    df
    .unique(subset=["bsID"])
    .select([
        "bsID",
        "bsName",
        "bsDescription",
        "bsPublic",
        "owner",
        "sort"
    ])
    .sort("bsID")
)
membership = (
    df
    .select(["facid", "bsID"])
    .unique()
    .sort(["facid", "bsID"])
)

In [6]:
facilities

facid,site,name,address,zone,owned,status,latitude,longitude
str,str,str,str,str,str,str,f64,f64
"""0000""","""CEN""","""FAC ID-NON-BLDG ORGS""","""NOT APPLICABLE""",null,null,"""OPEN""",null,null
"""0110""","""CEN""","""CALVIN HILL DAY CARE""","""HIGHLAND STREET, 150""","""ZONE 2""","""OO""","""OPEN""",41.328181,-72.920866
"""0120""","""CEN""","""WHITEHALL APTS 375-385""","""CANNER STREET, 375-385""","""ZONE 2""","""OO""","""OPEN""",41.325814,-72.921398
"""0122""","""CEN""","""WHITEHALL APTS 405-407""","""CANNER STREET, 405-407""","""ZONE 2""","""OO""","""OPEN""",41.326013,-72.922116
"""0124""","""CEN""","""WHITEHALL APTS 511-521""","""PROSPECT STREET, 511-521""","""ZONE 2""","""OO""","""OPEN""",41.326293,-72.921995
…,…,…,…,…,…,…,…,…
"""4245""","""WST""","""WC ENERGY SCIENCES CTR""","""WEST CAMPUS DRIVE, 520""",null,"""OO""","""OPEN""",41.258284,-72.990042
"""4250""","""WST""","""WC COLLECTION ST CTR""","""WEST CAMPUS DRIVE, 900""",null,"""OO""","""OPEN""",41.259342,-72.987395
"""4270""","""WST""","""WC STORAGE & RECEIV CTR""","""WEST CAMPUS DRIVE, 750""",null,"""OO""","""OPEN""",41.257826,-72.985888


In [7]:
building_sets

bsID,bsName,bsDescription,bsPublic,owner,sort
i64,str,null,i64,str,i64
2,"""Served By SPP""",null,1,"""sa""",60
3,"""Served By CPP""",null,1,"""sa""",60
4,"""Served By WPP""",null,1,"""sa""",60
5,"""Served By CPP or SPP""",null,1,"""sa""",60
20,"""Power Plants""",null,1,"""sa""",60
…,…,…,…,…,…
55,"""Academic & Admin: Small""",null,1,"""sa""",70
56,"""YSM Laboratories""",null,1,"""sa""",70
57,"""YSM Non Laboratories""",null,1,"""sa""",70


In [8]:
membership

facid,bsID
str,i64
"""0000""",59
"""0110""",59
"""0120""",59
"""0122""",59
"""0124""",59
…,…
"""4270""",58
"""4272""",4
"""4272""",21


In [9]:
# Save to CSV
facilities.write_csv("../data/raw/facilities.csv")
building_sets.write_csv("../data/raw/building_sets.csv")
membership.write_csv("../data/raw/membership.csv")